# Train And Benchmark

Thin orchestration notebook for `debug_tiny`, `debug_synthetic_generalization`, `debug_synthetic_generalization_large`, `local_small`, and `hg002_chr20_small`.
All project logic is run through the pinned conda interpreter via subprocess so the notebook itself stays import-light.


In [320]:
import json
import os
import subprocess
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint

PYTHON = '/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python'
NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_CWD if (NOTEBOOK_CWD / 'scripts').exists() else NOTEBOOK_CWD.parent

PRESET = 'debug_synthetic_generalization'
RUN_NAME = 'full'
RUN_FETCH = False
ENABLE_LOW_THRESHOLD_DEBUG = True
ENABLE_ARGMAX_DEBUG = True
ENABLE_OVERFIT_DEBUG = False
OVERFIT_RUN_NAME = 'target_only'
OVERFIT_NUM_EXAMPLES = 4
OVERFIT_CASES = ['sub', 'ins', 'del', 'copy']

CONFIG_PATH = PROJECT_ROOT / 'configs' / f'{PRESET}.yaml'
TEMP_CONFIG_PATH = PROJECT_ROOT / 'outputs' / 'tmp' / f'{PRESET}_active_debug.yaml'
TEMP_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CONFIG_PATH =', CONFIG_PATH)
print('PYTHON =', PYTHON)


PROJECT_ROOT = /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild
CONFIG_PATH = /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/configs/debug_synthetic_generalization.yaml
PYTHON = /Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python


In [321]:
def run_script(*args, check=True):
    command = [PYTHON, *args]
    print(' '.join(str(part) for part in command))
    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        env={**os.environ, 'PYTHONPATH': str(PROJECT_ROOT / 'src')},
        check=False,
        text=True,
        capture_output=True,
    )
    print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise subprocess.CalledProcessError(completed.returncode, command, completed.stdout, completed.stderr)
    return completed


if PRESET == 'debug_tiny':
    print('Running mandatory staged overfit suite before benchmark/debug runs...')
    run_script('scripts/run_overfit_suite.py', '--output-dir', 'outputs/mandatory_overfit')
    OVERFIT_SUITE_PATH = PROJECT_ROOT / 'outputs' / 'mandatory_overfit' / 'overfit_suite.json'
    pprint(json.loads(OVERFIT_SUITE_PATH.read_text(encoding='utf-8')))


make_config_args = [
    'scripts/make_debug_config.py',
    '--base-config', str(CONFIG_PATH),
    '--output-config', str(TEMP_CONFIG_PATH),
    '--preset', PRESET,
    '--run-name', RUN_NAME,
]
if ENABLE_LOW_THRESHOLD_DEBUG:
    make_config_args.append('--enable-low-threshold-debug')
if ENABLE_OVERFIT_DEBUG and PRESET == 'debug_tiny':
    make_config_args.extend([
        '--enable-overfit-debug',
        '--overfit-run-name', OVERFIT_RUN_NAME,
        '--overfit-num-examples', str(OVERFIT_NUM_EXAMPLES),
        '--overfit-cases', ','.join(OVERFIT_CASES),
    ])
elif ENABLE_OVERFIT_DEBUG:
    print(f'Skipping overfit-debug config rewrite for {PRESET}; using the graduated benchmark config as-is.')
run_script(*make_config_args)
active_config = json.loads(TEMP_CONFIG_PATH.read_text(encoding='utf-8'))
ACTIVE_CONFIG_PATH = TEMP_CONFIG_PATH
ACTIVE_RUN_NAME = active_config.get('debug_run_name', RUN_NAME)
OUTPUT_ROOT = PROJECT_ROOT / active_config['train']['output_dir']
RUN_OUTPUT_DIR = OUTPUT_ROOT / ACTIVE_RUN_NAME
print('ACTIVE_CONFIG_PATH =', ACTIVE_CONFIG_PATH)
print('ACTIVE_RUN_NAME =', ACTIVE_RUN_NAME)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
pprint(active_config)


/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/make_debug_config.py --base-config /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/configs/debug_synthetic_generalization.yaml --output-config /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/tmp/debug_synthetic_generalization_active_debug.yaml --preset debug_synthetic_generalization --run-name full --enable-low-threshold-debug
name: debug_synthetic_generalization
regression_targets:
  notebook_21_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_18_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_19_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
dataset:
  kind: synthetic
  synthetic_suite: harder
  sample_id: DEBUG
  outp

## Optional Fetch Subset


In [322]:
if RUN_FETCH:
    run_script('scripts/fetch_region_subset.py', '--config', str(ACTIVE_CONFIG_PATH))


## Preprocess, Train, Export


In [323]:
RUNS = ['no_edit', 'support_rule', 'consensus', 'target_only', 'full_hybrid', 'full_neural_only']

run_script('scripts/preprocess_dataset.py', '--config', str(ACTIVE_CONFIG_PATH))

if any(run_name in RUNS for run_name in ['no_edit', 'support_rule', 'consensus']):
    run_script('scripts/run_baselines.py', '--config', str(ACTIVE_CONFIG_PATH))

for learned_run in ['target_only', 'full']:
    if learned_run in RUNS or (learned_run == 'full' and any(run in RUNS for run in ['full_hybrid', 'full_neural_only'])):
        run_script('scripts/train_model.py', '--config', str(ACTIVE_CONFIG_PATH), '--run-name', learned_run)

run_script('scripts/export_summary.py', '--config', str(ACTIVE_CONFIG_PATH), '--warn-on-regression-failure')
summary_path = OUTPUT_ROOT / 'benchmark_summary.json'
print('summary_path =', summary_path)
if not summary_path.exists():
    raise FileNotFoundError(f'Expected benchmark summary was not created: {summary_path}')


/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/preprocess_dataset.py --config /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/tmp/debug_synthetic_generalization_active_debug.yaml
name: debug_synthetic_generalization
regression_targets:
  notebook_21_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_18_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_19_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
dataset:
  kind: synthetic
  synthetic_suite: harder
  sample_id: DEBUG
  output_dir: outputs/debug_synthetic_generalization/dataset
  splits:
    train: 68
    val: 17
    test: 17
  max_window_length: 64
  overlap: 8
  max_support_reads: 6
  max_deletion_length: 3
  synthetic_seed: 31
  shared_examp

## Compare Summaries


In [324]:
summary_path = OUTPUT_ROOT / 'benchmark_summary.json'
if not summary_path.exists():
    print('benchmark_summary.json missing; exporting summary now...')
    run_script('scripts/export_summary.py', '--config', str(ACTIVE_CONFIG_PATH), '--warn-on-regression-failure')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
pprint(summary)

gap_path = OUTPUT_ROOT / 'full' / 'hybrid_gap_summary.json'
if gap_path.exists():
    print('\nHybrid gap summary:')
    pprint(json.loads(gap_path.read_text(encoding='utf-8')))
else:
    print('\nHybrid gap summary missing; train full_hybrid/full_neural_only to create it.')


{'preset': 'debug_synthetic_generalization',
 'regression_targets': {'checks': [{'comparator': '>=',
                                    'expected': 0.9722,
                                    'metric': 'usable_score',
                                    'observed': 0.9722222222222223,
                                    'passed': True,
                                    'run_name': 'full_hybrid',
                                    'target': 'notebook_21_full_hybrid'},
                                   {'comparator': '<=',
                                    'expected': 0.0,
                                    'metric': 'hard_edit_false_positive_rate',
                                    'observed': 0.0,
                                    'passed': True,
                                    'run_name': 'full_hybrid',
                                    'target': 'notebook_21_full_hybrid'},
                                   {'comparator': '<=',
                                    'e

## Thresholded Decode Probe


In [325]:
thresholded_reports_by_checkpoint = {}
for checkpoint_name in ['best.ckpt', 'last.ckpt']:
    checkpoint_path = RUN_OUTPUT_DIR / checkpoint_name
    probe_dir = RUN_OUTPUT_DIR / checkpoint_name.replace('.ckpt', '')
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'thresholded',
        '--run-output-dir', str(probe_dir),
        '--example-filters', 'sub_,ins_,del_,copy_',
    )
    thresholded_reports_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'thresholded_probe.json').read_text(encoding='utf-8'))
pprint({name: reports[:2] for name, reports in thresholded_reports_by_checkpoint.items()})


/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/debug_probe.py --config /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/tmp/debug_synthetic_generalization_active_debug.yaml --checkpoint /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/debug_synthetic_generalization/full/best.ckpt --mode thresholded --run-output-dir /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/debug_synthetic_generalization/full/best --example-filters sub_,ins_,del_,copy_
name: debug_synthetic_generalization
regression_targets:
  notebook_21_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_18_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_19_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
 

## Direct Argmax-Edit Probe


In [326]:
if ENABLE_ARGMAX_DEBUG:
    argmax_reports_by_checkpoint = {}
    for checkpoint_name in ['best.ckpt', 'last.ckpt']:
        checkpoint_path = RUN_OUTPUT_DIR / checkpoint_name
        probe_dir = RUN_OUTPUT_DIR / checkpoint_name.replace('.ckpt', '')
        run_script(
            'scripts/debug_probe.py',
            '--config', str(ACTIVE_CONFIG_PATH),
            '--checkpoint', str(checkpoint_path),
            '--mode', 'argmax',
            '--run-output-dir', str(probe_dir),
            '--example-filters', 'sub_,ins_,del_,copy_',
        )
        argmax_reports_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'argmax_probe.json').read_text(encoding='utf-8'))
    pprint({name: reports[:2] for name, reports in argmax_reports_by_checkpoint.items()})


/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/debug_probe.py --config /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/tmp/debug_synthetic_generalization_active_debug.yaml --checkpoint /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/debug_synthetic_generalization/full/best.ckpt --mode argmax --run-output-dir /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/debug_synthetic_generalization/full/best --example-filters sub_,ins_,del_,copy_
name: debug_synthetic_generalization
regression_targets:
  notebook_21_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_18_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_19_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    ov

## Per-Class Learnability


In [327]:
classwise_probe_by_checkpoint = {}
for checkpoint_name in ['best.ckpt', 'last.ckpt']:
    checkpoint_path = RUN_OUTPUT_DIR / checkpoint_name
    probe_dir = RUN_OUTPUT_DIR / checkpoint_name.replace('.ckpt', '')
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'classwise',
        '--run-output-dir', str(probe_dir),
    )
    classwise_probe_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'classwise_probe.json').read_text(encoding='utf-8'))
pprint(classwise_probe_by_checkpoint)


/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/debug_probe.py --config /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/tmp/debug_synthetic_generalization_active_debug.yaml --checkpoint /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/debug_synthetic_generalization/full/best.ckpt --mode classwise --run-output-dir /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/debug_synthetic_generalization/full/best
name: debug_synthetic_generalization
regression_targets:
  notebook_21_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_18_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_19_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
dataset:


## Missed Hard-Edit Evidence


In [328]:
missed_evidence_by_checkpoint = {}
for checkpoint_name in ['best.ckpt', 'last.ckpt']:
    checkpoint_path = RUN_OUTPUT_DIR / checkpoint_name
    probe_dir = RUN_OUTPUT_DIR / checkpoint_name.replace('.ckpt', '')
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'missed_evidence',
        '--run-output-dir', str(probe_dir),
    )
    missed_evidence_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'missed_evidence_probe.json').read_text(encoding='utf-8'))
pprint({name: reports[:5] for name, reports in missed_evidence_by_checkpoint.items()})

false_sub_by_checkpoint = {}
ins_payload_by_checkpoint = {}
hybrid_gap_by_checkpoint = {}
hybrid_miss_by_checkpoint = {}
for checkpoint_name in ['best.ckpt', 'last.ckpt']:
    checkpoint_path = RUN_OUTPUT_DIR / checkpoint_name
    probe_dir = RUN_OUTPUT_DIR / checkpoint_name.replace('.ckpt', '')
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'false_sub',
        '--run-output-dir', str(probe_dir),
    )
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'ins_payload',
        '--run-output-dir', str(probe_dir),
    )
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'hybrid_gap',
        '--run-output-dir', str(probe_dir),
    )
    run_script(
        'scripts/debug_probe.py',
        '--config', str(ACTIVE_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--mode', 'hybrid_miss',
        '--run-output-dir', str(probe_dir),
    )
    false_sub_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'false_sub_probe.json').read_text(encoding='utf-8'))
    ins_payload_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'ins_payload_probe.json').read_text(encoding='utf-8'))
    hybrid_gap_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'hybrid_gap_probe.json').read_text(encoding='utf-8'))
    hybrid_miss_by_checkpoint[checkpoint_name] = json.loads((probe_dir / 'hybrid_miss_probe.json').read_text(encoding='utf-8'))
print('\nFalse SUB diagnostics:')
pprint(false_sub_by_checkpoint)
print('\nINS payload diagnostics:')
pprint(ins_payload_by_checkpoint)
print('\nHybrid gap diagnostics:')
pprint({name: report['summary'] for name, report in hybrid_gap_by_checkpoint.items()})
print('\nHybrid missed-edit diagnostics (SUB_T / boundary INS_A blockers live here):')
pprint(hybrid_miss_by_checkpoint)


/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/debug_probe.py --config /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/tmp/debug_synthetic_generalization_active_debug.yaml --checkpoint /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/debug_synthetic_generalization/full/best.ckpt --mode missed_evidence --run-output-dir /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/debug_synthetic_generalization/full/best
name: debug_synthetic_generalization
regression_targets:
  notebook_21_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_18_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
  notebook_19_full_hybrid:
    usable_score_min: 0.9722
    hard_edit_false_positive_rate_max: 0.0
    overcorrection_rate_max: 0.0
dat

## Scale Synthetic Test

This optional cell runs the larger multi-seed synthetic benchmark from inside the notebook. It is intentionally off by default so opening/running the notebook does not accidentally start a heavier experiment.


In [329]:
RUN_SCALE_SYNTHETIC = True
USE_SMALL_NOISY_SCALE = True  # fast curated noisy gate; set False for full noisy multiseed validation
SCALE_SEEDS = '47' if USE_SMALL_NOISY_SCALE else '47,48,49,50,51'
SCALE_RUNS = 'target_only,full'
SCALE_CONFIG_NAME = 'debug_synthetic_noisy_small' if USE_SMALL_NOISY_SCALE else 'debug_synthetic_generalization_large'
SCALE_OUTPUT_NAME = 'debug_synthetic_noisy_small_multiseed' if USE_SMALL_NOISY_SCALE else 'debug_synthetic_generalization_multiseed'
SCALE_BASE_CONFIG = PROJECT_ROOT / 'configs' / f'{SCALE_CONFIG_NAME}.yaml'
SCALE_OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / SCALE_OUTPUT_NAME


def load_scale_json(path):
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None


def print_scale_results(output_root):
    index_path = output_root / 'multiseed_index.json'
    audit_path = output_root / 'multiseed_audit.json'
    summary_path = output_root / 'multiseed_summary.json'
    index = load_scale_json(index_path)
    audit = load_scale_json(audit_path)
    summary = load_scale_json(summary_path)

    print('scale_output_root =', output_root)
    print('multiseed_index =', index_path)
    print('multiseed_audit =', audit_path)
    print('multiseed_summary =', summary_path)

    if summary:
        print('\nsummary:')
        pprint(summary)
        no_edit = summary.get('runs', {}).get('no_edit', {}).get('usable_score', {}).get('mean')
        full_hybrid = summary.get('runs', {}).get('full_hybrid', {}).get('usable_score', {}).get('mean')
        if no_edit is not None and full_hybrid is not None:
            print(f"\nusable_score mean: full_hybrid={full_hybrid:.4f} no_edit={no_edit:.4f} delta={full_hybrid - no_edit:.4f}")
    else:
        print('\nNo multiseed_summary.json found yet.')

    if audit:
        print('\naudit uniqueness:')
        pprint(audit.get('uniqueness', {}))
        print('\nseed-by-seed table:')
        pprint(audit.get('seed_table', []))
        print('\nfast noisy gate:')
        pprint(audit.get('fast_noisy_gate', []))
    else:
        print('No multiseed_audit.json found yet.')

    if index:
        print('\ndiagnostic dump paths:')
        for seed, row in index.items():
            full_dir = Path(row['output_dir']) / 'full'
            print(f"seed {seed}: false_hard={full_dir / 'false_hard_probe.json'}")
            print(f"seed {seed}: vetoed_true={full_dir / 'vetoed_true_probe.json'}")
            print(f"seed {seed}: sub_payload={full_dir / 'sub_payload_probe.json'}")
            print(f"seed {seed}: ins_payload={full_dir / 'ins_payload_probe.json'}")
            print(f"seed {seed}: hybrid_miss={full_dir / 'hybrid_miss_probe.json'}")
            print(f"seed {seed}: support_rule_audit={full_dir / 'support_rule_audit_probe.json'}")
            print(f"seed {seed}: rule_calibration={full_dir / 'rule_calibration_probe.json'}")
            print(f"seed {seed}: calibration_gap={full_dir / 'calibration_gap_probe.json'}")


print('Small noisy scale is selected:', USE_SMALL_NOISY_SCALE)
print('Scale config =', SCALE_BASE_CONFIG)
print('Scale seeds =', SCALE_SEEDS)
print('Scale runs =', SCALE_RUNS)

if RUN_SCALE_SYNTHETIC:
    run_script(
        'scripts/run_synthetic_multiseed.py',
        '--base-config', str(SCALE_BASE_CONFIG),
        '--output-root', str(SCALE_OUTPUT_ROOT),
        '--seeds', SCALE_SEEDS,
        '--runs', SCALE_RUNS,
    )
    print_scale_results(SCALE_OUTPUT_ROOT)
else:
    print('Scale synthetic test is configured but not run.')
    print('Set RUN_SCALE_SYNTHETIC = True in this cell to execute it.')
    print('Command that will run:')
    print(PYTHON, 'scripts/run_synthetic_multiseed.py', '--base-config', SCALE_BASE_CONFIG, '--output-root', SCALE_OUTPUT_ROOT, '--seeds', SCALE_SEEDS, '--runs', SCALE_RUNS)
    if (SCALE_OUTPUT_ROOT / 'multiseed_summary.json').exists():
        print('\nExisting results found:')
        print_scale_results(SCALE_OUTPUT_ROOT)


Small noisy scale is selected: True
Scale config = /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/configs/debug_synthetic_noisy_small.yaml
Scale seeds = 47
Scale runs = target_only,full
/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/run_synthetic_multiseed.py --base-config /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/configs/debug_synthetic_noisy_small.yaml --output-root /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/debug_synthetic_noisy_small_multiseed --seeds 47 --runs target_only,full
/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/preprocess_dataset.py --config /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/debug_synthetic_noisy_small_multiseed/seed_47/config.yaml
name: debug_synthetic_noisy_small_seed_47
regression_targets:
  fast_noisy_safety:
    run_name: full_hybrid
    usable_score_min: 

## Large-Noisy FP Audit

This is the next safety-focused run. It runs the large noisy seeds, dumps every false hard edit, categorizes the failure mechanisms, and prepares edit-type-specific allow-gate training rows. It does not change model code or tune thresholds inside the benchmark run.


In [330]:
RUN_LARGE_NOISY_FP_AUDIT = True
LARGE_FP_AUDIT_SEEDS = '47,48'
LARGE_FP_AUDIT_RUNS = 'target_only,full'
LARGE_FP_AUDIT_CONFIG = PROJECT_ROOT / 'configs' / 'debug_synthetic_generalization_large.yaml'
LARGE_FP_AUDIT_OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'large_noisy_fp_audit'

print('Large noisy FP audit config =', LARGE_FP_AUDIT_CONFIG)
print('Large noisy FP audit seeds =', LARGE_FP_AUDIT_SEEDS)
print('Large noisy FP audit output =', LARGE_FP_AUDIT_OUTPUT_ROOT)

if RUN_LARGE_NOISY_FP_AUDIT:
    run_script(
        'scripts/run_synthetic_multiseed.py',
        '--base-config', str(LARGE_FP_AUDIT_CONFIG),
        '--output-root', str(LARGE_FP_AUDIT_OUTPUT_ROOT),
        '--seeds', LARGE_FP_AUDIT_SEEDS,
        '--runs', LARGE_FP_AUDIT_RUNS,
    )
    print_scale_results(LARGE_FP_AUDIT_OUTPUT_ROOT)
else:
    print('Large noisy FP audit is configured but not run.')
    print('Set RUN_LARGE_NOISY_FP_AUDIT = True to execute it.')
    print(PYTHON, 'scripts/run_synthetic_multiseed.py', '--base-config', LARGE_FP_AUDIT_CONFIG, '--output-root', LARGE_FP_AUDIT_OUTPUT_ROOT, '--seeds', LARGE_FP_AUDIT_SEEDS, '--runs', LARGE_FP_AUDIT_RUNS)


Large noisy FP audit config = /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/configs/debug_synthetic_generalization_large.yaml
Large noisy FP audit seeds = 47,48
Large noisy FP audit output = /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/large_noisy_fp_audit
/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/run_synthetic_multiseed.py --base-config /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/configs/debug_synthetic_generalization_large.yaml --output-root /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/large_noisy_fp_audit --seeds 47,48 --runs target_only,full
/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/preprocess_dataset.py --config /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/large_noisy_fp_audit/seed_47/config.yaml
name: debug_synthetic_generalization_large_

## False-Edit Table And Mechanism Categories

This cell aggregates `false_hard_probe.json` across large noisy seeds into one flat table and summarizes whether errors are false `SUB`, `INS`, `DEL`, neighbor-induced, homopolymer deletion, boundary-related, support-rule forced, or neural hallucination not vetoed.


In [331]:
def read_json_or_none(path):
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None


def edit_type(label):
    if not label:
        return 'UNKNOWN'
    if label.startswith('SUB_'):
        return 'SUB'
    if label.startswith('INS_'):
        return 'INS'
    if label == 'DEL':
        return 'DEL'
    return 'COPY'


def flatten_false_edit(seed, row):
    evidence = row.get('support_evidence', {})
    trace = row.get('decoder_trace', {})
    confidence = trace.get('rule_confidence', {})
    predicted = row.get('predicted_label')
    support_rule = row.get('support_rule_label')
    neural_only = row.get('neural_only_label')
    support_rule_positive = support_rule not in (None, 'COPY')
    neural_agrees = neural_only == predicted
    mechanism_tags = list(row.get('mechanism_tags', []))
    if not mechanism_tags:
        family = edit_type(predicted)
        mechanism_tags.append(f'false_{family.lower()}')
        if support_rule_positive:
            mechanism_tags.append('support_rule_false_positive')
        if neural_agrees:
            mechanism_tags.append('neural_hallucination_not_vetoed')
        if row.get('forced_by_rule'):
            mechanism_tags.append('hybrid_forced_false_edit')
        if confidence.get('neighbor_edit_proximity', 0):
            mechanism_tags.append('neighbor_induced')
        if family == 'DEL' and confidence.get('homopolymer_run_length', 1) >= 4:
            mechanism_tags.append('homopolymer_false_deletion')
        if row.get('pos') in (0, None):
            mechanism_tags.append('boundary_ambiguous')
    return {
        'seed': int(seed),
        'example_id': row.get('example_id'),
        'position': row.get('position', row.get('pos')),
        'gold_label': row.get('gold_label'),
        'predicted_label': predicted,
        'support_rule_label': support_rule,
        'neural_only_label': neural_only,
        'edit_type': row.get('edit_type', edit_type(predicted)),
        'target_base': row.get('target_base', evidence.get('target_base')),
        'truth_base': row.get('truth_base', evidence.get('truth_base')),
        'support_base_counts': row.get('support_base_counts', evidence.get('support_base_counts')),
        'support_insertion_counts': row.get('support_insertion_counts', evidence.get('support_ins_base_counts')),
        'support_deletion_count': row.get('support_deletion_count', evidence.get('support_del_count')),
        'support_insertion_count': row.get('support_insertion_count', evidence.get('support_ins_count')),
        'support_depth': row.get('support_depth', confidence.get('support_depth')),
        'support_fraction': row.get('support_fraction', confidence.get('support_fraction')),
        'support_margin': row.get('support_margin', confidence.get('support_margin')),
        'entropy': row.get('entropy', confidence.get('local_entropy')),
        'homopolymer_flag': bool(row.get('homopolymer_flag', confidence.get('homopolymer_run_length', 1) >= 4)),
        'homopolymer_run_length': row.get('homopolymer_run_length', confidence.get('homopolymer_run_length')),
        'neighbor_edit_distance': row.get('neighbor_edit_distance'),
        'neighbor_edit_proximity': row.get('neighbor_edit_proximity', confidence.get('neighbor_edit_proximity')),
        'boundary_flag': bool(row.get('boundary_flag', row.get('pos') == 0)),
        'type_probs': row.get('type_probs'),
        'sub_base_probs': row.get('sub_base_probs'),
        'ins_base_probs': row.get('ins_base_probs'),
        'predicted_type_probability': row.get('predicted_type_probability'),
        'predicted_payload_probability': row.get('predicted_payload_probability'),
        'veto_or_rescue_reason': row.get('veto_or_rescue_reason', row.get('veto_status', [])),
        'candidate_label': row.get('candidate_label'),
        'likely_source': row.get('likely_source'),
        'forced_by_rule': bool(row.get('forced_by_rule')),
        'rescued_by_neural': bool(row.get('rescued_by_neural', trace.get('rescued_by_neural', False))),
        'rescued_by_support_payload': bool(row.get('rescued_by_support_payload', trace.get('rescued_by_support_payload', False))),
        'rescued_by_sub_t_calibration': bool(row.get('rescued_by_sub_t_calibration', trace.get('rescued_by_sub_t_calibration', False))),
        'mechanism_tags': mechanism_tags,
    }


def load_false_edit_table(output_root):
    output_root = Path(output_root)
    index = read_json_or_none(output_root / 'multiseed_index.json') or {}
    rows = []
    for seed, info in index.items():
        false_path = Path(info['output_dir']) / 'full' / 'false_hard_probe.json'
        for row in read_json_or_none(false_path) or []:
            rows.append(flatten_false_edit(seed, row))
    return rows


def summarize_false_edit_mechanisms(rows):
    tag_counts = Counter(tag for row in rows for tag in row['mechanism_tags'])
    by_type = Counter(row['edit_type'] for row in rows)
    by_seed = Counter(row['seed'] for row in rows)
    by_source = Counter(row['likely_source'] for row in rows)
    by_rule = Counter('rule_positive' if row['support_rule_label'] not in (None, 'COPY') else 'rule_negative' for row in rows)
    return {
        'total_false_edits': len(rows),
        'by_seed': dict(by_seed),
        'by_edit_type': dict(by_type),
        'by_likely_source': dict(by_source),
        'by_support_rule_status': dict(by_rule),
        'mechanism_tags': dict(tag_counts),
    }


large_false_edit_table = load_false_edit_table(LARGE_FP_AUDIT_OUTPUT_ROOT)
large_false_edit_summary = summarize_false_edit_mechanisms(large_false_edit_table)
LARGE_FP_AUDIT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(LARGE_FP_AUDIT_OUTPUT_ROOT / 'large_false_edit_table.json').write_text(json.dumps(large_false_edit_table, indent=2), encoding='utf-8')
(LARGE_FP_AUDIT_OUTPUT_ROOT / 'large_false_edit_mechanism_summary.json').write_text(json.dumps(large_false_edit_summary, indent=2), encoding='utf-8')

print('large_false_edit_table.json =', LARGE_FP_AUDIT_OUTPUT_ROOT / 'large_false_edit_table.json')
print('large_false_edit_mechanism_summary.json =', LARGE_FP_AUDIT_OUTPUT_ROOT / 'large_false_edit_mechanism_summary.json')
pprint(large_false_edit_summary)
print('\nFirst 10 false edits:')
pprint(large_false_edit_table[:10])


large_false_edit_table.json = /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/large_noisy_fp_audit/large_false_edit_table.json
large_false_edit_mechanism_summary.json = /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/large_noisy_fp_audit/large_false_edit_mechanism_summary.json
{'by_edit_type': {'DEL': 9},
 'by_likely_source': {'hybrid_forced_by_support_rule': 4,
                      'support_rule_positive_but_not_forced': 5},
 'by_seed': {47: 5, 48: 4},
 'by_support_rule_status': {'rule_positive': 9},
 'mechanism_tags': {'boundary_ambiguous': 1,
                    'false_del': 9,
                    'homopolymer_false_deletion': 3,
                    'hybrid_forced_false_edit': 4,
                    'low_support_margin': 9,
                    'neural_hallucination_not_vetoed': 9,
                    'neural_rescue_false_edit': 5,
                    'support_rule_false_positive': 9},
 'total_false_edits': 9}

Fi

## Edit-Type-Specific Allow Gate Prep

This cell prepares an offline candidate-edit gate dataset. Positives come from support-rule true positives in the small noisy slice; negatives come from large noisy false hard edits. It trains/tunes separate `SUB`, `INS`, and `DEL` toy gates in-notebook for analysis only, selecting thresholds under a zero-false-positive constraint.


In [332]:
ALLOW_GATE_SMALL_ROOT = PROJECT_ROOT / 'outputs' / 'debug_synthetic_noisy_small_multiseed'
ALLOW_GATE_LARGE_ROOT = LARGE_FP_AUDIT_OUTPUT_ROOT


def family_from_candidate(row):
    label = row.get('support_rule_label') if row.get('support_rule_label') not in (None, 'COPY') else row.get('predicted_label')
    return edit_type(label)


def gate_features(row):
    support_rule = row.get('support_rule_label')
    predicted = row.get('predicted_label', row.get('hybrid_label'))
    neural = row.get('neural_only_label')
    return {
        'support_fraction': float(row.get('support_fraction') or 0.0),
        'support_margin': float(row.get('support_margin') or 0.0),
        'entropy': float(row.get('entropy') or 0.0),
        'support_depth': float(row.get('support_depth') or 0.0),
        'payload_confidence': float(row.get('payload_confidence', row.get('predicted_payload_probability') or 0.0) or 0.0),
        'type_confidence': float(row.get('neural_type_prob', row.get('predicted_type_probability') or 0.0) or 0.0),
        'rule_neural_agree': float(support_rule not in (None, 'COPY') and support_rule == neural),
        'rule_candidate_agree': float(support_rule not in (None, 'COPY') and support_rule == predicted),
        'homopolymer_flag': float(bool(row.get('homopolymer_flag', row.get('homopolymer_run_length', 1) >= 4))),
        'neighbor_ambiguity_flag': float(bool(row.get('neighbor_edit_proximity', 0) or row.get('neighbor_edit_distance') == 1)),
        'boundary_flag': float(bool(row.get('boundary_flag', row.get('read_boundary_position', False)))),
        'del_fraction': float(row.get('del_fraction') or 0.0),
        'support_rule_positive': float(support_rule not in (None, 'COPY')),
    }


def support_rule_rows_from_root(output_root):
    output_root = Path(output_root)
    index = read_json_or_none(output_root / 'multiseed_index.json') or {}
    rows = []
    for seed, info in index.items():
        audit = read_json_or_none(Path(info['output_dir']) / 'full' / 'support_rule_audit_probe.json') or {}
        for row in audit.get('rows', []):
            updated = dict(row)
            updated['seed'] = int(seed)
            updated['source'] = 'small_noisy_support_rule'
            updated['target_allow'] = bool(row.get('is_true_positive'))
            updated['candidate_family'] = family_from_candidate(updated)
            updated['features'] = gate_features(updated)
            rows.append(updated)
    return rows


def large_false_rows_for_gate(rows):
    output = []
    for row in rows:
        updated = dict(row)
        updated['source'] = 'large_noisy_false_edit'
        updated['target_allow'] = False
        updated['candidate_family'] = family_from_candidate(updated)
        updated['features'] = gate_features(updated)
        output.append(updated)
    return output


def score_gate_row(row):
    # Transparent analysis score: favor strong support + confident payload/type, penalize ambiguity.
    f = row['features']
    return (
        1.5 * f['support_fraction']
        + 0.15 * f['support_margin']
        + 0.8 * f['payload_confidence']
        + 0.4 * f['type_confidence']
        + 0.3 * f['rule_candidate_agree']
        - 0.5 * f['entropy']
        - 0.35 * f['neighbor_ambiguity_flag']
        - 0.25 * f['homopolymer_flag']
        - 0.15 * f['boundary_flag']
    )


def zero_fp_threshold_for_rows(rows):
    if not rows:
        return None
    scored = [(score_gate_row(row), bool(row['target_allow'])) for row in rows]
    thresholds = sorted({score for score, _ in scored}, reverse=True) + [max(score for score, _ in scored) + 1.0]
    best = {'threshold': thresholds[-1], 'allowed': 0, 'allowed_true': 0, 'allowed_false': 0, 'blocked_true': sum(label for _, label in scored)}
    for threshold in thresholds:
        allowed = [(score, label) for score, label in scored if score >= threshold]
        fp = sum(1 for _, label in allowed if not label)
        tp = sum(1 for _, label in allowed if label)
        if fp == 0 and tp >= best['allowed_true']:
            best = {
                'threshold': round(float(threshold), 4),
                'allowed': len(allowed),
                'allowed_true': tp,
                'allowed_false': fp,
                'blocked_true': sum(1 for row_score, label in scored if label and row_score < threshold),
            }
    return best


GATE_FEATURE_NAMES = [
    'support_fraction',
    'support_margin',
    'entropy',
    'support_depth',
    'payload_confidence',
    'type_confidence',
    'rule_neural_agree',
    'rule_candidate_agree',
    'homopolymer_flag',
    'neighbor_ambiguity_flag',
    'boundary_flag',
    'del_fraction',
    'support_rule_positive',
]


def train_logistic_zero_fp_gate(rows, family):
    if len(rows) < 4 or len({bool(row['target_allow']) for row in rows}) < 2:
        return {'status': 'not_enough_mixed_rows', 'family': family, 'fallback_score_gate': zero_fp_threshold_for_rows(rows)}
    import torch

    x = torch.tensor([[row['features'][name] for name in GATE_FEATURE_NAMES] for row in rows], dtype=torch.float32)
    y = torch.tensor([float(row['target_allow']) for row in rows], dtype=torch.float32)
    mean = x.mean(dim=0)
    std = x.std(dim=0).clamp(min=1e-6)
    xz = (x - mean) / std
    torch.manual_seed(17)
    model = torch.nn.Linear(xz.shape[1], 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
    # False positives are expensive: weight negatives slightly higher while fitting the audit gate.
    weights = torch.where(y > 0.5, torch.ones_like(y), torch.full_like(y, 2.0))
    for _ in range(500):
        optimizer.zero_grad()
        logits = model(xz).squeeze(-1)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y, weight=weights)
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        scores = torch.sigmoid(model(xz).squeeze(-1)).tolist()
    thresholds = sorted({float(score) for score in scores}, reverse=True) + [1.01]
    best = {'threshold': 1.01, 'allowed': 0, 'allowed_true': 0, 'allowed_false': 0, 'blocked_true': int(y.sum().item())}
    for threshold in thresholds:
        allowed = [idx for idx, score in enumerate(scores) if score >= threshold]
        fp = sum(1 for idx in allowed if not rows[idx]['target_allow'])
        tp = sum(1 for idx in allowed if rows[idx]['target_allow'])
        if fp == 0 and tp >= best['allowed_true']:
            best = {
                'threshold': round(float(threshold), 4),
                'allowed': len(allowed),
                'allowed_true': tp,
                'allowed_false': fp,
                'blocked_true': sum(1 for idx, row in enumerate(rows) if row['target_allow'] and scores[idx] < threshold),
            }
    weights_by_feature = {
        name: round(float(weight), 4)
        for name, weight in zip(GATE_FEATURE_NAMES, model.weight.detach().squeeze(0).tolist())
    }
    scored_examples = []
    for row, score in zip(rows, scores):
        scored_examples.append({
            'source': row.get('source'),
            'seed': row.get('seed'),
            'example_id': row.get('example_id'),
            'position': row.get('position', row.get('pos')),
            'candidate_family': family,
            'candidate_label': row.get('support_rule_label') if row.get('support_rule_label') not in (None, 'COPY') else row.get('predicted_label'),
            'target_allow': bool(row['target_allow']),
            'allow_probability': round(float(score), 4),
            'zero_fp_allows': float(score) >= best['threshold'],
        })
    return {
        'status': 'ok',
        'family': family,
        'feature_names': GATE_FEATURE_NAMES,
        'zero_fp': best,
        'weights': weights_by_feature,
        'bias': round(float(model.bias.detach().item()), 4),
        'scored_examples': scored_examples,
        'fallback_score_gate': zero_fp_threshold_for_rows(rows),
    }


small_gate_rows = support_rule_rows_from_root(ALLOW_GATE_SMALL_ROOT)
large_gate_rows = large_false_rows_for_gate(large_false_edit_table)
allow_gate_rows = small_gate_rows + large_gate_rows
gate_summary = {}
for family in ['SUB', 'INS', 'DEL']:
    family_rows = [row for row in allow_gate_rows if row.get('candidate_family') == family]
    gate_summary[family] = {
        'rows': len(family_rows),
        'true_allow': sum(1 for row in family_rows if row['target_allow']),
        'false_block': sum(1 for row in family_rows if not row['target_allow']),
        'zero_fp_threshold': zero_fp_threshold_for_rows(family_rows),
        'logistic_zero_fp_gate': train_logistic_zero_fp_gate(family_rows, family),
    }

(ALLOW_GATE_LARGE_ROOT / 'allow_gate_training_rows.json').write_text(json.dumps(allow_gate_rows, indent=2), encoding='utf-8')
(ALLOW_GATE_LARGE_ROOT / 'allow_gate_summary.json').write_text(json.dumps(gate_summary, indent=2), encoding='utf-8')
print('allow_gate_training_rows.json =', ALLOW_GATE_LARGE_ROOT / 'allow_gate_training_rows.json')
print('allow_gate_summary.json =', ALLOW_GATE_LARGE_ROOT / 'allow_gate_summary.json')
pprint(gate_summary)


allow_gate_training_rows.json = /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/large_noisy_fp_audit/allow_gate_training_rows.json
allow_gate_summary.json = /Users/shanejayasundera/LRS-Error-Correction/new_solution/omega_lr_rebuild/outputs/large_noisy_fp_audit/allow_gate_summary.json
{'DEL': {'false_block': 9,
         'logistic_zero_fp_gate': {'fallback_score_gate': {'allowed': 0,
                                                           'allowed_false': 0,
                                                           'allowed_true': 0,
                                                           'blocked_true': 0,
                                                           'threshold': 3.7312},
                                   'family': 'DEL',
                                   'status': 'not_enough_mixed_rows'},
         'rows': 9,
         'true_allow': 0,
         'zero_fp_threshold': {'allowed': 0,
                               'allowed_false': 0,

## Mandatory Overfit Gate

The notebook now runs `scripts/run_overfit_suite.py` before benchmark/debug training when `PRESET = 'debug_tiny'`.
That gate proves:

- single-example `SUB` overfit
- single-example `INS` overfit
- single-example `DEL` overfit
- mixed 4-example overfit


In [333]:
print('ENABLE_OVERFIT_DEBUG =', ENABLE_OVERFIT_DEBUG)
print('OVERFIT_RUN_NAME =', OVERFIT_RUN_NAME)
print('OVERFIT_NUM_EXAMPLES =', OVERFIT_NUM_EXAMPLES)
print('OVERFIT_CASES =', OVERFIT_CASES)


ENABLE_OVERFIT_DEBUG = False
OVERFIT_RUN_NAME = target_only
OVERFIT_NUM_EXAMPLES = 4
OVERFIT_CASES = ['sub', 'ins', 'del', 'copy']
